In [1]:
!rm -rf /content/yolov1-cupy
!git clone https://github.com/mihnea-popescu/yolov1-cupy.git

import sys
sys.path.append('/content/yolov1-cupy')

from main import TestClass

test = TestClass()
test.test()          # IT'S WORKING!

Cloning into 'yolov1-cupy'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 85 (delta 34), reused 63 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 51.32 KiB | 1.90 MiB/s, done.
Resolving deltas: 100% (34/34), done.
Github classes initialized!


Downloading the ImageNet10 dataset from kaggle

In [2]:
!curl -L -o /content/imagenet10.zip https://www.kaggle.com/api/v1/datasets/download/liusha249/imagenet10
print("Extracting dataset (quiet unzip)...")
!unzip -q /content/imagenet10.zip -d /content/yolov1-cupy
print("Done.")
!rm /content/imagenet10.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1304M  100 1304M    0     0   149M      0  0:00:08  0:00:08 --:--:--  162M
Extracting dataset (quiet unzip)...
Done.


## Process Images

In [3]:
from image_batch_loader import image_label_batch

batch_size = 8
x, y_cpu = image_label_batch("/content/yolov1-cupy", batch_size=batch_size)
print("x.shape:", x.shape, "dtype:", x.dtype)
print("labels:", y_cpu)

x.shape: (8, 3, 224, 224) dtype: float64
labels: [8 2 6 9 8 5 3 0]


## Mini Darknet Structure

| Block | Layers                                                           | Output size   |
|-------|------------------------------------------------------------------|--------------|
| Input | —                                                                | 3 × 224 × 224|
| 1     | Conv2D(3→16, 3, pad=1) → BN → LeakyReLU → MaxPool(2)            | 16 × 112 × 112|
| 2     | Conv2D(16→32, 3, pad=1) → BN → LeakyReLU → MaxPool(2)           | 32 × 56 × 56 |
| 3     | Conv2D(32→64, 3, pad=1) → BN → LeakyReLU → MaxPool(2)           | 64 × 28 × 28 |
| 4     | Conv2D(64→128, 3, pad=1) → BN → LeakyReLU → MaxPool(2)          | 128 × 14 × 14|
| 5     | Conv2D(128→256, 3, pad=1) → BN → LeakyReLU → MaxPool(2)         | 256 × 7 × 7  |
| Head  | AvgPool(7) → Flatten → Linear(256, 10)                          | 10           |

In [ ]:
from mini_darknet import MiniDarknet

model = MiniDarknet(num_classes=10)
logits = model.forward(x)
print("logits.shape:", logits.shape, "dtype:", logits.dtype)
print(logits)

logits.shape: (8, 10) dtype: float64
